In [5]:
pip install pyvi

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
from datasets import load_dataset
from pyvi import ViTokenizer # Thay đổi theo yêu cầu để khớp PhoBERT
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix
import re
import time # Để đo Inference Time

# 1. Tải dữ liệu từ Hugging Face
print("Đang tải dữ liệu...")
dataset = load_dataset("pqthinh232/vietnamese-restaurant-review-sentiment-dataset")

df_train = pd.DataFrame(dataset['train'])
df_val = pd.DataFrame(dataset['validation'])
df_test = pd.DataFrame(dataset['test'])

# 2. Hàm tiền xử lý chuẩn hóa (ViTokenizer + Truncation)
def preprocess_text(text):
    if not isinstance(text, str): return ""
    text = text.lower()
    
    
    # Sử dụng ViTokenizer để đồng bộ với PhoBERT
    text = ViTokenizer.tokenize(text)
    
    # Giới hạn độ dài câu: Cắt ở mốc 150 từ (theo yêu cầu 150-200)
    words = text.split()
    if len(words) > 256:
        words = words[:256]
    
    return " ".join(words)

print("Đang tiền xử lý văn bản chuẩn hóa (ViTokenizer & Truncation)...")
df_train['clean_review'] = df_train['review'].apply(preprocess_text)
df_val['clean_review'] = df_val['review'].apply(preprocess_text)
df_test['clean_review'] = df_test['review'].apply(preprocess_text)

# 3. Vector hóa TF-IDF
print("Đang chuyển đổi TF-IDF...")
tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=10000)
X_train = tfidf.fit_transform(df_train['clean_review'])
X_val = tfidf.transform(df_val['clean_review'])
X_test = tfidf.transform(df_test['clean_review'])

y_train = df_train['label']
y_test = df_test['label']

# 4. Huấn luyện SVM
print("Đang huấn luyện SVM (class_weight='balanced')...")
# Đã có class_weight='balanced' để giải quyết tỉ lệ 60-20-20
model = SVC(kernel='linear', C=1.0, class_weight='balanced', probability=True)
model.fit(X_train, y_train)

# 5. Đánh giá và Đo Inference Time (QUAN TRỌNG)
print("\n--- ĐÁNH GIÁ MÔ HÌNH (BENCHMARK) ---")

# Đo thời gian dự đoán trên tập Test
start_inference = time.time()
y_pred = model.predict(X_test)
end_inference = time.time()

inference_time = end_inference - start_inference
avg_time_per_sample = inference_time / len(df_test)

# Tính Macro F1
macro_f1 = f1_score(y_test, y_pred, average='macro')

print(f"1. Tổng thời gian dự đoán (Inference Time) trên {len(df_test)} mẫu: {inference_time:.4f} giây")
print(f"2. Thời gian trung bình mỗi mẫu: {avg_time_per_sample:.6f} giây")
print(f"3. Macro F1-score: {macro_f1:.4f}")
print(f"4. Accuracy: {accuracy_score(y_test, y_pred):.4f}")

# Hiển thị Confusion Matrix 3x3
print("\n[Confusion Matrix]")
cm = confusion_matrix(y_test, y_pred)
print(cm)

print("\n[Chi tiết Classification Report]")
print(classification_report(y_test, y_pred, target_names=['Tiêu cực (0)', 'Tích cực (1)', 'Trung tính (2)']))

# 6. Hàm dự đoán câu mới
def predict_sentiment(text):
    processed = preprocess_text(text)
    vectorized = tfidf.transform([processed])
    prediction = model.predict(vectorized)[0]
    mapping = {0: "Tiêu cực", 1: "Tích cực", 2: "Trung tính"}
    return mapping[prediction]

c:\Users\ACER\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Đang tải dữ liệu...


c:\Users\ACER\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ACER\.cache\huggingface\hub\datasets--pqthinh232--vietnamese-restaurant-review-sentiment-dataset. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating test split: 100%|██████████| 3272/3272 [00:00

Đang tiền xử lý văn bản chuẩn hóa (ViTokenizer & Truncation)...
Đang chuyển đổi TF-IDF...
Đang huấn luyện SVM (class_weight='balanced')...

--- ĐÁNH GIÁ MÔ HÌNH (BENCHMARK) ---
1. Tổng thời gian dự đoán (Inference Time) trên 3272 mẫu: 27.5331 giây
2. Thời gian trung bình mỗi mẫu: 0.008415 giây
3. Macro F1-score: 0.7994
4. Accuracy: 0.8246

[Confusion Matrix]
[[ 532   11   79]
 [  34 1663  284]
 [  47  119  503]]

[Chi tiết Classification Report]
                precision    recall  f1-score   support

  Tiêu cực (0)       0.87      0.86      0.86       622
  Tích cực (1)       0.93      0.84      0.88      1981
Trung tính (2)       0.58      0.75      0.66       669

      accuracy                           0.82      3272
     macro avg       0.79      0.82      0.80      3272
  weighted avg       0.85      0.82      0.83      3272



In [ ]:
import joblib
import os
from huggingface_hub import HfApi, login

# --- CẤU HÌNH THÔNG TIN ---
# Dán token của bạn vào đây (Token phải có quyền WRITE)
HF_TOKEN = "" 
# Thay 'your-username' bằng tên thật của bạn
REPO_NAME = "neidythedev/vietnamese-restaurant-sentiment-svm" 

# --- BƯỚC 1: LƯU MÔ HÌNH VÀ TF-IDF RA FILE TẠM ---
print("1. Đang lưu mô hình ra file local...")
model_name = "svm_sentiment_model.pkl"
tfidf_name = "tfidf_vectorizer.pkl"

joblib.dump(model, model_name)
joblib.dump(tfidf, tfidf_name)

# Tạo file requirements.txt
with open("requirements.txt", "w") as f:
    f.write("scikit-learn\nunderthesea\njoblib\npandas")

# --- BƯỚC 2: ĐĂNG NHẬP (GÁN CỨNG TOKEN) ---
print("2. Đang đăng nhập vào Hugging Face...")
login(token=HF_TOKEN)

# --- BƯỚC 3: TẠO REPO VÀ PUSH FILE ---
api = HfApi()

print(f"3. Đang tạo/kiểm tra repository: {REPO_NAME}...")
api.create_repo(repo_id=REPO_NAME, repo_type="model", exist_ok=True)

# Danh sách các file cần upload
files_to_upload = [model_name, tfidf_name, "requirements.txt"]

print("4. Đang upload các file lên Hugging Face Hub...")
for file in files_to_upload:
    try:
        api.upload_file(
            path_or_fileobj=file,
            path_in_repo=file,
            repo_id=REPO_NAME,
            token=HF_TOKEN # Đảm bảo quyền ghi
        )
        print(f"   - Đã upload thành công: {file}")
    except Exception as e:
        print(f"   - Lỗi khi upload {file}: {e}")

print(f"\n--- HOÀN TẤT! ---")
print(f"Link mô hình: https://huggingface.co/{REPO_NAME}")

1. Đang lưu mô hình ra file local...
2. Đang đăng nhập vào Hugging Face...
3. Đang tạo/kiểm tra repository: neidythedev/vietnamese-restaurant-sentiment-svm...
4. Đang upload các file lên Hugging Face Hub...


Processing Files (1 / 1): 100%|██████████| 11.4MB / 11.4MB,  331kB/s  
New Data Upload: 100%|██████████| 11.4MB / 11.4MB,  330kB/s  


   - Đã upload thành công: svm_sentiment_model.pkl


Processing Files (1 / 1): 100%|██████████|  403kB /  403kB,  336kB/s  
New Data Upload: 100%|██████████|  403kB /  403kB,  336kB/s  


   - Đã upload thành công: tfidf_vectorizer.pkl
   - Đã upload thành công: requirements.txt

--- HOÀN TẤT! ---
Link mô hình: https://huggingface.co/neidythedev/vietnamese-restaurant-sentiment-svm
